# COMP0178 - Lesson 6: Transaction Management

Source: [Lecture 6: Transaction Management](../Lectures/COMP0178%206%20Transaction%20Management.pdf), **80 PDF pages** (Weeks 6-7). All page numbers below refer to this PDF. Companion practical: [Lab 6 notes](../labs/lab6.ipynb).

**Main idea:** a transaction groups related operations so that concurrent work and failures do not leave the database in a state that the application considers invalid. Concurrency control decides which operations may proceed; recovery makes committed work durable and removes incomplete work.

- **Lecture content** is explained in the numbered sections, with PDF-page references.
- **Clarifications** identify simplified or misleading slide wording.
- **Supplementary practice** uses standard Python and disposable SQLite, with worked answers and checks. It illustrates concepts; SQLite does not implement every lock/timestamp protocol taught here.
- Read each practice prompt, fill the answer cell if desired, then compare with the supplied solution. The notebooks do not require a MySQL server or additional Python packages.

## 1. Why a transaction is a unit of work (PDF pages 1-8)

- A business operation can require several database writes. Transferring GBP 20 from account A to account B requires both a debit and a credit. Either change alone violates the intended invariant that the total money is unchanged.
- **A system failure** can occur after the debit but before the credit, or after commit while some changed pages remain only in memory buffers.
- **Concurrency** means several transactions are in progress together. Interleaving their reads and writes can produce an outcome that neither intended.
- A transaction boundary tells the DBMS which statements must commit or roll back together. `COMMIT` accepts the transaction; `ROLLBACK` discards its unfinished effects. How transactions begin and whether each statement autocommits depends on the DBMS and connection configuration: merely placing `COMMIT` after already-autocommitted statements cannot make them atomic retrospectively.

### Three motivating anomalies

| Problem | Example | Why it is wrong |
| --- | --- | --- |
| Partial transfer | A changes from 100 to 80; failure prevents B changing from 30 to 50 | Total becomes 110 instead of 130 |
| Lost update | Withdrawal reads 100; deposit reads 100; writes are 80 then 120 | Correct serial balance is 100; the withdrawal is overwritten |
| Inconsistent analysis | A report reads A=100, transfer commits A=80/B=50, report then reads B=50 | Report says 150 although the total is always 130 |

**Slide clarifications:** page 6's final credit should write the **payee's** balance. Page 8 demonstrates a mixed-time/inconsistent analysis: this need not be a dirty read if the transfer committed before B was read, and it need not be a nonrepeatable read because the same item is not necessarily read twice. A **dirty read** specifically reads another transaction's uncommitted write; a **nonrepeatable read** gets different values when rereading an item because another transaction changed it.

## 2. ACID, states, and DBMS responsibilities (PDF pages 9-14)

| Property | Meaning | Transfer example |
| --- | --- | --- |
| Atomicity | All effects of the transaction happen, or none survive its abort | Undo a debit if the credit cannot complete |
| Consistency | A correct transaction takes a valid database state to another valid state | Preserve total balance and relevant constraints |
| Isolation | Concurrent work is controlled so transactions do not observe unacceptable interference | Prevent a report from using a forbidden partial transfer |
| Durability | Once commit is acknowledged, the effects survive the failures covered by the system's recovery guarantees | Recover the committed transfer after a crash |

- Consistency also depends on correct application logic and declared constraints. Serializing an incorrect transfer does not fix its logic.
- Full serializable isolation aims to make concurrent execution behave like some serial execution; weaker isolation levels permit specified anomalies. The ACID slogan does not mean every database connection uses the strongest isolation automatically.
- **Active/uncommitted:** work is in progress and can be aborted. **Committed:** work has been accepted and is recovered if lost. **Aborted:** the effects must be removed.
- A later **compensating transaction** can reverse a business outcome after commit. That is a new committed action, rather than silently treating the original committed transaction as aborted.
- Page 11's **uncommitted dependency** occurs when T2 uses a value written by T1 before T1 commits; an abort of T1 can invalidate T2 too.

### Components (page 13)

- **Transaction manager:** accepts and tracks transaction requests and their boundaries.
- **Scheduler / concurrency controller / lock manager:** coordinates concurrent operations.
- **Recovery manager:** uses durable recovery information after failures and aborts.
- **Buffer manager:** moves database pages between memory and permanent storage. A committed change in memory and an on-disk database page are not the same thing.

## 3. Runnable SQL practice: transfer, commit, and rollback

**Supplementary SQLite exercise.** Starting from A=100 and B=30, transfer 20 and check that the total remains 130. Then deliberately transfer to a nonexistent account: the debit must be rolled back. Write the transaction boundaries and the invariant before running the solution.

- SQLite examples below use Python's `sqlite3` with `isolation_level=None`, so `BEGIN`, `COMMIT`, and `ROLLBACK` explicitly control each transaction.
- The code uses an in-memory database; rerunning the setup starts fresh. SQLite's transaction syntax is not a claim about MySQL/Oracle defaults.

**Python API note (verified 18 November 2026):** this exercise uses legacy transaction control, the default in the tested Python 3.12 environment. In that mode, `isolation_level=None` disables implicit transaction starts; the explicit SQL controls the boundary. Python 3.12+ also provides a separate `autocommit` option, which can override this behaviour. See the [official Python 3.12 transaction-control documentation](https://docs.python.org/3.12/library/sqlite3.html#transaction-control-via-the-isolation-level-attribute).

In [1]:
# Your answer: which statements belong inside the same transaction?
# What balances should remain after a transfer to a missing account fails?

In [2]:
import sqlite3

bank = sqlite3.connect(":memory:", isolation_level=None)
bank.executescript("""
CREATE TABLE Account (
    account_id INTEGER PRIMARY KEY,
    balance INTEGER NOT NULL CHECK (balance >= 0)
);
INSERT INTO Account VALUES (1, 100), (2, 30);
""")

def balances():
    return bank.execute("SELECT account_id, balance FROM Account ORDER BY account_id").fetchall()

def transfer(payer, payee, amount):
    if amount <= 0 or payer == payee:
        raise ValueError("Choose different accounts and a positive amount")
    bank.execute("BEGIN")
    try:
        debit = bank.execute(
            "UPDATE Account SET balance = balance - ? "
            "WHERE account_id = ? AND balance >= ?",
            (amount, payer, amount),
        )
        if debit.rowcount != 1:
            raise ValueError("Missing payer or insufficient balance")
        credit = bank.execute(
            "UPDATE Account SET balance = balance + ? WHERE account_id = ?",
            (amount, payee),
        )
        if credit.rowcount != 1:
            raise ValueError("Missing payee")
        bank.execute("COMMIT")
    except Exception:
        bank.execute("ROLLBACK")
        raise

transfer(1, 2, 20)
assert balances() == [(1, 80), (2, 50)]
assert bank.execute("SELECT SUM(balance) FROM Account").fetchone()[0] == 130
before_failed_transfer = balances()
try:
    transfer(1, 999, 10)
except ValueError as exc:
    print("Failed transfer rolled back:", exc)
assert balances() == before_failed_transfer
print("Committed balances:", balances())
bank.close()

Failed transfer rolled back: Missing payee
Committed balances: [(1, 80), (2, 50)]


## 4. Schedules and serializability (PDF pages 15-22)

- A **schedule** lists operations from several transactions while preserving the operation order within each transaction.
- A **serial schedule** completes one transaction before starting another. An **interleaved schedule** mixes their operations.
- A schedule is **serializable** when it has an appropriate equivalence to **some** serial execution. It need not match every serial order: for example, deleting a staff member and reassigning a property can have different legitimate outcomes in different orders (page 18).
- Reasoning assumes each transaction is correct when run alone and relevant integrity constraints are enforced. Serializability addresses interference; it is not itself the whole ACID guarantee.

### The lecture's simple scheduling policy (pages 20-21)

- Allow read-only transactions to overlap.
- Allow transactions accessing disjoint data to overlap.
- Delay conflicting transactions that overlap on data with a possible write.
- This is easy to enforce but conservative. More detailed conflict analysis can permit useful interleavings that this whole-transaction policy rejects.

Notation: `r1(A)` = T1 reads A; `w2(A)` = T2 writes A; `c1` = T1 commits; `a1` = T1 aborts. A **blind write** writes an item without first reading it in that transaction.

## 5. Conflict serializability and precedence graphs (PDF pages 22-25)

Two operations conflict if they belong to **different transactions**, access the **same item**, and **at least one is a write**.

| Earlier operation | Later operation | Edge |
| --- | --- | --- |
| `w1(A)` | `r2(A)` | T1 -> T2 |
| `r1(A)` | `w2(A)` | T1 -> T2 |
| `w1(A)` | `w2(A)` | T1 -> T2 |
| `r1(A)` | `r2(A)` | No edge: two reads do not conflict |

1. Draw one node per transaction.
2. Scan all relevant ordered operation pairs; add an edge from the earlier operation's transaction to the later one's for every conflict.
3. A **cycle** means the schedule is **not conflict serializable**.
4. An **acyclic graph** means it is conflict serializable; any topological order gives an equivalent serial order. A graph can have more than one valid order.

### Lecture exercise: page 24

```text
r1(A), r2(A), w1(A), w2(A)

r1(A) before w2(A): T1 -> T2
r2(A) before w1(A): T2 -> T1
w1(A) before w2(A): T1 -> T2

Graph: T1 <-> T2     Result: cycle, not conflict serializable.
```

- Do not infer serializability merely from the final numerical values in one example. Conflict equivalence preserves the relevant read/write ordering structurally.
- Page 25 proposes aborting transactions to break cycles. A victim choice must break every remaining cycle; recovery must also undo invalid effects and handle dependent transactions. Checking after irreversible commits would be too late.

### Runnable graph exercise

Predict the edges before running the checker. In the second schedule T2 accesses B while T1 accesses A, but T2 later reads T1's A. Is it serial, serializable, both, or neither?

In [3]:
# Your answer for the page-24 graph:
# Your answer for the interleaved second schedule:

In [4]:
from collections import defaultdict

# Operations are triples: (transaction, "r" or "w", item).
def precedence_graph(schedule):
    graph = {transaction: set() for transaction, _, _ in schedule}
    for index, (t1, op1, item1) in enumerate(schedule):
        for t2, op2, item2 in schedule[index + 1:]:
            if t1 != t2 and item1 == item2 and "w" in (op1, op2):
                graph[t1].add(t2)
    return graph

def topological_order(graph):
    nodes = set(graph) | {v for targets in graph.values() for v in targets}
    indegree = {node: 0 for node in nodes}
    for targets in graph.values():
        for target in targets:
            indegree[target] += 1
    ready = sorted(node for node in nodes if indegree[node] == 0)
    result = []
    while ready:
        node = ready.pop(0)
        result.append(node)
        for target in sorted(graph.get(node, set())):
            indegree[target] -= 1
            if indegree[target] == 0:
                ready.append(target)
                ready.sort()
    return result if len(result) == len(nodes) else None

page24 = [("T1", "r", "A"), ("T2", "r", "A"),
          ("T1", "w", "A"), ("T2", "w", "A")]
interleaved = [("T1", "r", "A"), ("T2", "r", "B"),
               ("T1", "w", "A"), ("T2", "w", "B"), ("T2", "r", "A")]
assert precedence_graph(page24) == {"T1": {"T2"}, "T2": {"T1"}}
assert topological_order(precedence_graph(page24)) is None
assert topological_order(precedence_graph(interleaved)) == ["T1", "T2"]
for label, schedule in [("Page 24", page24), ("Interleaved", interleaved)]:
    graph = precedence_graph(schedule)
    print(label, graph, "serial order:", topological_order(graph))

Page 24 {'T1': {'T2'}, 'T2': {'T1'}} serial order: None
Interleaved {'T1': {'T2'}, 'T2': set()} serial order: ['T1', 'T2']


## 6. View serializability and blind writes (PDF pages 26-28)

View equivalence preserves three facts between schedules:

- A read of an **initial value** must still read that initial value.
- Each read of a value produced by a transaction must still read from the **same producing transaction**.
- Each item must have the **same final writer**. Matching a coincidental final numeric value alone is not enough.

### Page 26's example

```text
Interleaved: r1(x), w2(x), w1(x), w3(x)
Serial:      r1(x), w1(x), w2(x), w3(x)    [T1, T2, T3]
```

- In both schedules T1 reads the initial x, and T3 is the final writer. T2 and T3 perform blind writes, so no extra reads must be preserved.
- The interleaved schedule has T1 -> T2 from `r1(x)` before `w2(x)`, but T2 -> T1 from `w2(x)` before `w1(x)`: it is **not conflict serializable**.
- It is nevertheless **view serializable**, as the displayed serial schedule demonstrates.
- Every conflict-serializable schedule is view serializable; the reverse implication fails. A view-serializable schedule outside the conflict-serializable class requires a blind write.
- Testing general view serializability is NP-complete (page 28), making conflict-based or protocol-based methods more practical.

In [5]:
# Supplementary check of the specific read-from/final-writer example.
# This is a signature check for fixed, straight-line schedules, not a general
# implementation of a DBMS or a polynomial-time view-serializability test.
def view_signature(schedule):
    last_writer = {}
    reads = []
    for transaction, operation, item in schedule:
        if operation == "r":
            reads.append((transaction, item, last_writer.get(item, "INITIAL")))
        else:
            last_writer[item] = transaction
    return reads, last_writer

blind_interleaving = [("T1", "r", "x"), ("T2", "w", "x"),
                      ("T1", "w", "x"), ("T3", "w", "x")]
blind_serial = [("T1", "r", "x"), ("T1", "w", "x"),
                ("T2", "w", "x"), ("T3", "w", "x")]
assert view_signature(blind_interleaving) == view_signature(blind_serial)
assert topological_order(precedence_graph(blind_interleaving)) is None
print("Same view signature:", view_signature(blind_interleaving))

Same view signature: ([('T1', 'x', 'INITIAL')], {'x': 'T3'})


## 7. Recoverability is a separate requirement (PDF pages 29-33)

If T2 actually **reads a value written by T1**, then T2 depends on T1. If T1 aborts, T2 may have to abort too. It must not already have made an irreversible commit.

- **Recoverable:** for every reads-from dependency T1 -> T2, `c1` occurs before `c2`. T2 may read before T1 commits, but must wait to commit. If T1 aborts, T2 must abort.
- **Nonrecoverable:** T2 commits before its source T1 commits; a later T1 abort creates a conflict between undoing invalid dependent work and preserving committed work.
- **Cascading rollback:** aborting one transaction forces aborts of transactions that read its uncommitted effects, possibly transitively.

**Supplementary distinctions used in exam practice:**

| Class | Restriction | What it prevents |
| --- | --- | --- |
| Recoverable | Reader commits after the transaction it read from commits | Committing a dependency on an uncommitted source |
| Cascadeless | Read another transaction's write only after that writer commits | Dirty reads and read-induced cascading aborts |
| Strict | After T1 writes x, no other transaction reads or writes x until T1 ends | Dirty reads and dirty writes; simplifies undo |

Strict implies cascadeless, which implies recoverable. These properties do not by themselves guarantee serializability.

### Worked traces

```text
w1(x), r2(x), c2, a1          Nonrecoverable: T2 committed a dirty dependency.
w1(x), r2(x), c1, c2          Recoverable, but not cascadeless.
w1(x), c1, r2(x), c2          Cascadeless (and strict for these accesses).
```

The first trace has an acyclic conflict graph T1 -> T2, so **conflict serializability alone does not ensure recoverability**. Conversely, the page-24 cyclic schedule can append `c1,c2` without any dirty reads because both reads came before the writes: recoverability alone does not ensure serializability.

**Clarification of page 31:** apply the commit rule to the **actual writer whose version the read observed**, rather than every transaction that happened to write that item earlier.

### Conceptual answer cell: classify the schedule

For `w1(x), r2(x), w2(y), r3(y), c1, c2, c3`, identify every reads-from dependency. Is it recoverable? Can aborting T1 before its commit cause cascading rollback?

In [6]:
# Your explanation:
# Dependencies:
# Recoverable or not, and why:
# Cascading aborts if T1 fails before c1:

<details><summary>Worked answer</summary>

- T2 reads from T1 on x; T3 reads from T2 on y. Dependencies are T1 -> T2 -> T3.
- The displayed commit order is `c1 < c2 < c3`, so it is recoverable.
- It is not cascadeless: both dependent reads occur before the source commits.
- If T1 aborts before c1, both T2 and T3 must be aborted if their reads have occurred and their effects depend on those values. A commit-order rule must stop them committing first.
</details>

## 8. Shared and exclusive locks; two-phase locking (PDF pages 34-40)

- A **shared/read lock (S)** allows other S locks on the same item, but conflicts with another transaction's exclusive lock.
- An **exclusive/write lock (X)** conflicts with both S and X locks held by other transactions.
- An **upgrade** changes S to X when compatible; a **downgrade** changes X to S. Upgrades acquire stronger access and belong in a 2PL growing phase; downgrades release access and begin the shrinking phase.

| Requested / held by another transaction | S held | X held |
| --- | --- | --- |
| S requested | Compatible | Wait/conflict |
| X requested | Wait/conflict | Wait/conflict |

### Why locking alone is insufficient (page 37)

T1 writes x and unlocks it; T2 reads/writes x and then reads/writes y; T1 then locks and reads/writes y. This produces T1 -> T2 on x and T2 -> T1 on y. T1 released x and later acquired y, allowing a cycle.

### Protocols and their guarantees

- **Basic two-phase locking (2PL):** a growing phase acquires locks; after the first release/downgrade, the shrinking phase acquires no new locks. This guarantees conflict serializability if all relevant accesses obey the protocol.
- 2PL does **not** mean that every lock is acquired at transaction start, nor that there are only two lock operations.
- **Strict 2PL** in the usual terminology retains X locks until commit/abort. This produces strict schedules and prevents dirty reads/writes while allowing some earlier release of S locks under the 2PL rules.
- **Rigorous 2PL** retains **all S and X locks** until commit/abort. This is the all-locks-at-end version named on page 39.
- **Conservative/static 2PL** obtains the whole required lock set atomically before the transaction proceeds: if the whole set is unavailable, do not retain a partial set while waiting. This avoids deadlock from incremental lock acquisition, but requires knowing the lock set and can reduce concurrency (page 43).

**Correction to page 39:** basic 2PL alone does not guarantee recoverability. `w1(x), unlock1(x), r2(x), c2, a1` can obey basic 2PL but is nonrecoverable. Retaining write locks until transaction end addresses this.

### Cascading rollback: page 40

T14 writes `bal_x` and releases its lock before finishing; T15 reads that uncommitted value and writes a derived value, also releasing its lock. T14's rollback invalidates T15. The abbreviated diagram shows T16's read-lock request but omits the read itself; if T16 consumes T15's derived uncommitted value, it must also roll back. Rigorous/strict locking keeps the uncommitted write inaccessible until T14 commits or its undo completes.

![Original lecture cascading-rollback diagram, PDF page 40](../assets/slide-images/Transaction_Management-page-040.png)

## 9. Deadlock, starvation, and wait-for graphs (PDF pages 41-44)

### A feasible deadlock trace

```text
T1 obtains X(A).             T2 obtains X(B).
T1 requests X(B): waits.     T2 requests X(A): waits.

Wait-for graph: T1 -> T2 -> T1.
```

- A **wait-for graph** has an edge from the **waiting transaction** to the **lock holder blocking it**. Under the lecture's ordinary lock-wait model, a cycle indicates deadlock.
- A **precedence graph** instead records ordering constraints from already scheduled conflicting operations. It answers a different question. A conflict cycle is not itself proof that a lock deadlock is currently occurring.
- **Starvation** is indefinite postponement of a transaction while others continue. The slides use “livelock” for this situation; more generally livelock can describe repeated activity/retries without useful progress.

### Three approaches

| Approach | Action | Trade-off |
| --- | --- | --- |
| Timeout | Abort/restart a transaction that waits too long | Simple; long legitimate waits can be mistaken for deadlock |
| Prevention | Enforce rules that make circular waiting impossible | May abort work before a deadlock actually forms |
| Detection and recovery | Periodically detect a wait-for cycle and abort a victim | Allows more waiting but pays detection/rollback cost |

- Victim selection can consider age, amount of work done, resources held, and previous restarts. Always choosing the same transaction can starve it.
- A victim must release its locks only as its rollback protocol safely permits, so other transactions cannot consume invalid data.

### Timestamp-based deadlock prevention (page 43)

“Older” means smaller timestamp. Here the timestamp chooses who waits; these are still **locking** schemes.

| Requester versus holder | Wait-die | Wound-wait |
| --- | --- | --- |
| Older requester wants younger holder's lock | Older requester waits | Abort (“wound”) the younger holder |
| Younger requester wants older holder's lock | Abort (“die”) the younger requester | Younger requester waits |

Restarts retain the original age in these schemes to reduce starvation. Wait-die wait edges run old -> young; wound-wait wait edges run young -> old, so neither can form a waiting cycle. This differs from the lecture's timestamp-ordering restart with a **new** timestamp.

In [7]:
# Supplementary wait-for graph exercise: the requests below both block.
holder = {"A": "T1", "B": "T2"}
requests = [("T1", "B"), ("T2", "A")]
wait_for = {"T1": set(), "T2": set()}
for requester, item in requests:
    if holder[item] != requester:
        wait_for[requester].add(holder[item])
assert wait_for == {"T1": {"T2"}, "T2": {"T1"}}
assert topological_order(wait_for) is None
print("Deadlock cycle:", wait_for)

Deadlock cycle: {'T1': {'T2'}, 'T2': {'T1'}}


## 10. Timestamp ordering and the obsolete-write rule (PDF pages 45-51)

Assign each transaction a timestamp `TS(T)`; the intended serial order is increasing timestamp order. For each item x track:

- `RTS(x)`: the **largest** timestamp of an accepted reader of x.
- `WTS(x)`: the timestamp of the latest accepted writer of x.

Initialize both to a value older than the transactions. “Latest” here means timestamp information, not blindly the last wall-clock operation: update read timestamps using `max` so an older accepted reader cannot erase knowledge of a newer reader.

| Attempt | Test | Action |
| --- | --- | --- |
| `read_T(x)` | `TS(T) < WTS(x)` | Abort/restart T: it is too old to read this version |
| `read_T(x)` | Otherwise | Accept; `RTS(x) = max(RTS(x), TS(T))` |
| `write_T(x)` | `TS(T) < RTS(x)` | Abort/restart T: a later transaction already read the old value |
| `write_T(x)` | Otherwise, `TS(T) < WTS(x)` | Lecture page 50: ignore the obsolete write |
| `write_T(x)` | Neither violation | Accept; set `WTS(x) = TS(T)` |

- The page-50 “ignore obsolete write” variant is commonly called **Thomas's write rule**. In **basic timestamp ordering**, the too-old write would instead cause abort/restart.
- **Check the read-timestamp violation first.** An obsolete write cannot simply be ignored if it would invalidate a newer transaction's earlier read.
- The lecture restarts rejected transactions with a new timestamp (page 47). No lock waits means no lock deadlock, but repeated restarts can still be costly or cause starvation.
- Timestamp order controls serializability; additional treatment of uncommitted versions/commit order is needed for recoverability.

### The lecture's lost-update attempt (pages 48 and 51)

For `TS(T1)=1`, `TS(T2)=2`, initially `RTS(A)=WTS(A)=0`:

| Step | Attempt | Metadata after accepted operation / decision |
| --- | --- | --- |
| 1 | T1 reads A=100 | RTS=1, WTS=0 |
| 2 | T2 reads A=100 | RTS=2, WTS=0 |
| 3 | T1 tries to write A=80 | Reject: 1 < RTS=2; abort T1 |

Do not continue as though T1's rejected write occurred. A full real recovery implementation must undo any earlier writes of an aborted transaction. Timestamp ordering alone does not guarantee recoverable commit ordering; avoid exposing/committing invalid dependencies as well.

### Runnable timestamp trace

Predict the first rejection in the lost-update schedule. Then compare `w2(x), w1(x)` under basic timestamp ordering and the lecture's obsolete-write variant.

In [8]:
# Your prediction:
# First rejected operation and the inequality:
# For w2(x), w1(x), should T1 be aborted or its write ignored under each rule?

In [9]:
def timestamp_trace(schedule, timestamps, *, ignore_obsolete=True):
    """Teaching trace of metadata checks; stop on first abort.

    No database values/log recovery are modelled. Use fresh metadata per call.
    """
    metadata = defaultdict(lambda: {"RTS": 0, "WTS": 0})
    trace = []
    for step, (transaction, operation, item) in enumerate(schedule, 1):
        ts = timestamps[transaction]
        state = metadata[item]
        if operation == "r":
            if ts < state["WTS"]:
                decision = "ABORT: TS < WTS"
            else:
                state["RTS"] = max(state["RTS"], ts)
                decision = "READ accepted"
        elif ts < state["RTS"]:
            decision = "ABORT: TS < RTS"
        elif ts < state["WTS"]:
            decision = "WRITE ignored" if ignore_obsolete else "ABORT: TS < WTS"
        else:
            state["WTS"] = ts
            decision = "WRITE accepted"
        trace.append((step, transaction, operation, item, decision, dict(state)))
        if decision.startswith("ABORT"):
            break
    return trace

ts = {"T1": 1, "T2": 2}
lost_update_trace = timestamp_trace(page24, ts)
assert lost_update_trace[-1][0] == 3
assert lost_update_trace[-1][4] == "ABORT: TS < RTS"
obsolete = [("T2", "w", "x"), ("T1", "w", "x")]
assert timestamp_trace(obsolete, ts)[-1][4] == "WRITE ignored"
assert timestamp_trace(obsolete, ts, ignore_obsolete=False)[-1][4] == "ABORT: TS < WTS"
# An older accepted read must not lower an existing RTS.
older_read = [("T2", "r", "x"), ("T1", "r", "x"), ("T1", "w", "x")]
assert timestamp_trace(older_read, ts)[-1][4] == "ABORT: TS < RTS"
for row in lost_update_trace:
    print(row)
print("Obsolete-write variant:", timestamp_trace(obsolete, ts)[-1])

(1, 'T1', 'r', 'A', 'READ accepted', {'RTS': 1, 'WTS': 0})
(2, 'T2', 'r', 'A', 'READ accepted', {'RTS': 2, 'WTS': 0})
(3, 'T1', 'w', 'A', 'ABORT: TS < RTS', {'RTS': 2, 'WTS': 0})
Obsolete-write variant: (2, 'T1', 'w', 'x', 'WRITE ignored', {'RTS': 0, 'WTS': 2})


## 11. Multiversion and optimistic methods; comparing protocols (PDF pages 52-54)

### Multiversion timestamp ordering (page 52)

- Store several versions of an item, with timestamp metadata. A transaction can read the version appropriate to its logical time rather than being forced to read a newer ineligible value.
- This can let readers proceed without the read aborts of the lecture's single-version algorithm, provided a suitable version is retained. Writes still require conflict checks.
- Trade-off: more storage, version metadata, and cleanup. “Multiple versions” is a technique; it does not by itself identify the isolation level of every DBMS that uses it.

### What the comparison diagram means (page 53)

- Conflict serializability is contained within view serializability; many schedules satisfy neither.
- 2PL and timestamp protocols generate **subsets** of acceptable schedules rather than accepting every possible serializable interleaving.
- Locking and timestamping permit different, overlapping sets. The timestamp set can extend beyond conflict serializability when the obsolete-write rule is used, while remaining view serializable under that protocol's assumptions.
- There is therefore no general inference that every schedule rejected by one protocol is inherently nonserializable.

### Optimistic concurrency control (page 54)

1. **Read/work phase:** access data and keep tentative changes in a private workspace.
2. **Validation:** test whether overlapping transactions' read/write sets create a forbidden conflict under the validation policy.
3. **Write phase:** publish the changes only after validation succeeds.

The lecture tracks start, validation-start, and finish times. In particular, a transaction that committed after this one began may have changed data this one read; an unfinished overlapping writer also requires careful validation. On failure, discard the private work and restart.

- Useful when conflicts are uncommon: less blocking overhead for uncontended work.
- Private tentative changes prevent other transactions reading these uncommitted changes, avoiding that source of cascading rollback.
- Under contention, a long transaction may do substantial work only to fail validation. Optimistic does not mean “skip concurrency checks.”

## 12. Granularity and intention locks (PDF pages 55-58)

- **Granularity** is the size of the locked data unit: database, file/table, page, record, or field.
- **Coarse locks** need fewer lock records and less bookkeeping, but block more potentially independent work.
- **Fine locks** allow more concurrent independent accesses, but need more memory and lock-management effort.

A containment hierarchy can be represented as:

```text
Database
  +-- Staff table
  |     +-- page -> record -> field
  +-- Property table
        +-- page -> record -> field
```

- A lock on a parent covers the relevant descendants. **Intention locks** on ancestors announce that a descendant is or will be locked, so a request for a conflicting coarse lock can be checked without scanning every descendant lock.
- An intention lock is not the same as exclusively locking the whole ancestor. Compatible work in a different branch can continue.
- Page 57: T1 S-locks Staff and marks its intent at database level. T2 can X-lock a suitable Property page with the corresponding ancestor intention locks. A request for an incompatible database-wide lock must wait; another compatible Staff read need not wait simply because Staff is already S-locked.
- The end-of-concurrency summary (page 58) brings together serializability **and** safe commit ordering. Recovery and correct transaction logic are also required for the complete ACID outcome.

## 13. Failures, logs, and backups (PDF pages 59-65, 68)

- **Loss of main memory:** volatile buffers disappear in a crash. Permanent database pages may survive, but represent a mixture of older committed data and incomplete work.
- **Loss of permanent storage:** restore a database backup and use retained logs to reapply later committed work. Keep backups/logs sufficiently independent of the failed storage; a backup on the only failed disk is not useful.
- A transaction may commit before all its changed data pages reach disk. Its **durable log records** can support redo.

### What a recovery log records

- Transaction ID, time/order, operation, affected item, and relevant **before-image** (old value) / **after-image** (new value).
- Transaction start, commit/abort, and checkpoint information.
- Links such as previous/next record pointers for following a transaction's operations.
- Insert/delete need the appropriate images: an inserted row has no earlier row to restore; a deleted row needs its old contents to undo deletion.

### Reading the lecture's log (pages 65 and 69)

- T1 starts at 10:12, updates `STAFF SL21` at 10:13, and commits at 10:18.
- T2 starts at 10:14, inserts `STAFF SG37`, deletes `STAFF SA9`, and updates `PROPERTY PG16`; it commits at 10:19.
- T3 starts at 10:18, inserts `PROPERTY PG4` at 10:20, and commits at 10:21.
- A checkpoint is printed at 10:19 with T2 and T3 active. **Log-record order matters:** it precedes the T2 commit record even though they share the displayed minute.
- If a crash occurs after the last shown commit, no shown transaction is a loser. Under the lecture's checkpoint model, T1's committed work is already durable; T2 and T3 are redo candidates after the checkpoint.
- If the crash occurs after T3's insert but before its commit record, T3 is an uncommitted loser: immediate-update recovery must remove its insert; T2 remains committed.

Logs consume storage and I/O and may become a bottleneck, but provide the evidence needed to distinguish accepted work from incomplete work.

## 14. Checkpoints and transaction classification (PDF pages 66-69)

The lecture uses a **simplified forced checkpoint**:

- Flush the current buffers to permanent storage.
- Record the checkpoint and which transactions are active.
- After a crash, identify transactions with durable commits and transactions still incomplete. Redo committed work that may not have reached the database, and undo uncommitted work that may have reached it.

![Checkpoint timeline from lecture page 67](../assets/slide-images/Transaction_Management-page-067.png)

### Solve the page-67 timeline

| Transactions | Where they end relative to checkpoint/failure | Recovery classification in the lecture model |
| --- | --- | --- |
| T2, T3 | Commit before checkpoint | Committed effects forced to disk: no redo required from this crash |
| T4 | Starts before checkpoint; commits after it but before failure | Redo candidate |
| T5 | Starts after checkpoint; commits before failure | Redo candidate |
| T1 | Starts before checkpoint; still active at failure | Undo under immediate update |
| T6 | Starts after checkpoint; still active at failure | Undo under immediate update |

- T1's pre-checkpoint writes may have been flushed by the checkpoint **without committing T1**. Undo may therefore require records from before the checkpoint. A checkpoint does not automatically make every earlier log record discardable.
- “Redo candidate” means committed work after the checkpoint must be considered; some corresponding pages may already be on disk. Redo is designed to be safe even when a change is already reflected there.
- **Supplementary qualification:** real systems may use fuzzy/non-forcing checkpoints and page-level metadata. Use the lecture's forced-buffer assumptions for this diagram rather than importing an unrelated recovery algorithm.

## 15. Deferred update, immediate update, and WAL (PDF pages 70-72)

| Technique in the lecture | Can uncommitted changes reach database pages? | Recovery after a crash |
| --- | --- | --- |
| Deferred update | No: apply database updates after commit point | No undo of uncommitted database changes; redo committed updates if needed |
| Immediate update | Yes: buffers may flush before commit | Undo uncommitted changes; redo committed changes if needed |

### Deferred update (page 71)

- Keep changes pending rather than writing uncommitted updates into the database.
- An abort before commit leaves no uncommitted database-page changes to undo.
- Redo may still be needed: committing does not imply every changed database page was forced to disk.

### Immediate update (page 72)

- An uncommitted page can be written to disk, so record **before-images** for undo.
- A committed changed page may still be only in memory, so record **after-images** for redo.
- Undo a transaction's updates **in reverse order**. If it changed C from 30 to 45 and then 45 to 55, restoring 45 then 30 gets the original value; restoring in the forward direction can leave 45.

### Write-ahead logging (WAL)

- The relevant log information must become durable **before the corresponding changed data page is written to permanent storage**.
- A commit must have the required durable log records, including its commit record, before success is acknowledged, so committed work can be recovered.
- “Write ahead” describes ordering between log durability and data-page writes; it does not require every committed data page to be on disk immediately.
- **Supplementary storage-policy vocabulary:** allowing uncommitted pages onto disk is often called **steal**; permitting commit without forcing all changed pages is **no-force**. Under the lecture's immediate-update/steal/no-force model, both undo and redo are needed. The exact requirements depend on the chosen storage policy.

### Runnable recovery exercise

A crash leaves A=100, B=50, C=55 on disk. T1 committed a transfer A:100->80 and B:30->50. T2, uncommitted, changed C:30->45->55. Explain why both undo and redo are needed and predict the recovered values.

In [10]:
# Your answer:
# Winner(s), loser(s), and recovery order:
# Expected A, B, C after recovery:

In [11]:
# Simplified immediate-update recovery for this independent-item example.
# Real recovery also handles page metadata, interleavings, compensation logs,
# crashes during recovery, and the concurrency protocol's guarantees.
log_updates = [
    ("T1", "A", 100, 80),
    ("T1", "B", 30, 50),
    ("T2", "C", 30, 45),
    ("T2", "C", 45, 55),
]
committed = {"T1"}
crash_disk = {"A": 100, "B": 50, "C": 55}

def recover_example(disk):
    restored = dict(disk)
    # Reverse order for loser undo.
    for transaction, item, before, after in reversed(log_updates):
        if transaction not in committed:
            restored[item] = before
    # Forward order for committed redo.
    for transaction, item, before, after in log_updates:
        if transaction in committed:
            restored[item] = after
    return restored

recovered = recover_example(crash_disk)
assert recovered == {"A": 80, "B": 50, "C": 30}
assert recover_example(recovered) == recovered
print("Crash disk:", crash_disk)
print("Recovered:", recovered)

Crash disk: {'A': 100, 'B': 50, 'C': 55}
Recovered: {'A': 80, 'B': 50, 'C': 30}


## 16. Shadow paging (PDF page 73)

- Start with a **shadow page table** describing the committed database and a **current page table** initially pointing to the same pages.
- On modification, write changed data to new pages and update the current mapping; keep the old shadow mapping/pages intact.
- On commit, make the new mapping the committed one through a safe atomic switch after the required new pages are durable. On abort, discard the tentative mapping and retain the shadow state.
- This is presented as a logless recovery alternative. It can avoid per-update undo/redo logging in the simple design, but new-page allocation, fragmentation, unused-page garbage collection, and safe publication of the mapping still need management.
- A page table is a mapping to data pages; changing the mapping is not the same as copying every database page for each write.

![Shadow-paging diagram, lecture PDF page 73](../assets/slide-images/Transaction_Management-page-073.png)

## 17. Complex and nested transactions (PDF pages 74-78)

- Long or collaborative work may touch many items and systems, hold resources for a long time, fail partway through, and limit concurrency.
- The lecture's travel-booking example combines flights, hotel, and rental car; flights may themselves have several legs and airlines.
- Break work into relatively independent subtransactions and run independent parts concurrently, while keeping a top-level outcome that coordinates success/failure.
- A **nested transaction** has parent/child structure. A child's success is **tentative**, conditional on the top-level transaction eventually committing.
- A parent can define how to handle a failed child, for example retrying that component or aborting the relevant larger unit.
- In the nested model described here, effects are isolated from **external transactions** until the whole top-level transaction commits, while suitable internal subparts can share them.
- Top-level ACID is still the goal: breaking work into parts does not by itself permit a hotel charge to survive an overall failed atomic booking.
- **Supplementary distinction:** real-world distributed workflows may instead use committed steps and compensating transactions. That is a different model from the lecture's tentative closed nested commits; do not assume a compensation is always identical to a rollback.

![Nested travel-booking transactions, lecture PDF page 77](../assets/slide-images/Transaction_Management-page-077.png)

## 18. Oracle case study and whole-lecture map (PDF pages 79-80)

**The lecture's case study, rather than current product setup instructions:**

- Oracle uses multiversion read consistency to give queries a consistent view; ordinary consistent reads are designed not to wait for a writer's row lock.
- Statement/transaction boundaries determine the unit of atomicity. Group statements correctly when the business operation needs to be atomic as a whole.
- The DBMS manages locks and detects deadlocks, and provides backup/recovery facilities. Manual lock control is possible when the application needs it.
- Page 79's “reads are never blocked” is a teaching shorthand for its multiversion consistent-read discussion, not a universal claim that every read-like operation, locking read, metadata access, or resource wait is impossible.

**Supplementary Oracle qualification:** in the default read-committed mode, each ordinary query has a statement-level consistent view. A serializable or read-only transaction uses a transaction-level view; several queries do not automatically share one snapshot merely because they are in the same default-mode transaction. [Oracle Database 26 concurrency and consistency documentation](https://docs.oracle.com/en/database/oracle/oracle-database/26/cncpt/data-concurrency-and-consistency.html) (checked 18 November 2026).

### Connect the parts

```text
Correct transaction logic and business invariants
                      |
             Transaction boundaries
                      |
      +---------------+----------------+
      |                                |
Concurrency control                 Recovery
Schedules, locks,             Durable logs, checkpoints,
timestamps, validation              backups
      |                                |
      +---------------+----------------+
                      |
         Consistent committed outcome
```

- Ask separately: **Is the interleaving serializable? Is the commit order recoverable? What is durable at failure? What can safely be undone/redone?**
- A graph answers only the question it was constructed for. ACID is achieved by the combined design, not by one graph or one slogan.

## 19. Common mistakes to catch

- Treating every pair of operations as a conflict: same item, different transactions, and at least one write are all required.
- Reversing an edge: precedence edges follow earlier -> later conflicting operation; wait-for edges follow waiter -> blocker.
- Giving an arbitrary serial order without checking all precedence edges.
- Declaring a conflict cycle “not serializable in any sense”: it establishes not **conflict** serializable; blind-write examples may still be view serializable.
- Using commit order to decide conflict serializability, or using an acyclic conflict graph to claim recoverability.
- Assuming basic 2PL holds all locks until commit or guarantees recoverability.
- Ignoring `TS < RTS` because an attempted write is also obsolete.
- Lowering RTS after an older accepted reader; RTS records the maximum accepted reader timestamp.
- Continuing a schedule as though an aborted/rejected operation succeeded.
- Assuming checkpoint commits active transactions or permits discarding all older undo information.
- Redoing an uncommitted loser's work in the simple model, undoing a winner, or applying one transaction's before-images in forward order.
- Confusing a durable commit log record with every affected database page already having been flushed.

## 20. Revision questions with answers

<details><summary>1. Why does a bank transfer need a transaction?</summary>

Debit and credit must form one atomic unit; otherwise a crash or application error can leave only one side applied. Isolation also protects concurrent work from unsuitable partial/interleaved effects.
</details>

<details><summary>2. What does a cycle in a precedence graph establish?</summary>

There is no serial transaction order preserving every conflicting operation order, so the schedule is not conflict serializable. It does not alone answer view serializability or current lock deadlock.
</details>

<details><summary>3. Why can two reads of the same item overlap?</summary>

Neither changes the value the other reads, so they do not conflict in the read/write conflict model; S locks are compatible.
</details>

<details><summary>4. Can a conflict-serializable schedule be nonrecoverable?</summary>

Yes: `w1(x), r2(x), c2, a1` has only T1 -> T2 in its precedence graph, but T2 commits after reading T1's uncommitted write and before T1 commits.
</details>

<details><summary>5. How do strict and rigorous 2PL differ?</summary>

Under the common naming used here, strict 2PL holds X locks until transaction end; rigorous 2PL holds S and X locks until then. Basic 2PL only separates lock acquisition from release.
</details>

<details><summary>6. Is every waiting transaction deadlocked?</summary>

No. A holder may finish and release the required lock normally. Deadlock requires a circular wait in the lecture's wait-for model; timeout is only a heuristic for deciding a wait is too long.
</details>

<details><summary>7. Older T1 requests a lock held by younger T2: what do wait-die and wound-wait do?</summary>

Wait-die allows T1 to wait. Wound-wait aborts the younger holder T2 so T1 can proceed after safe rollback/release.
</details>

<details><summary>8. Why does the lost-update timestamp example abort T1 at its write?</summary>

T2 has already read the old A, setting RTS(A)=2. T1's timestamp is 1, so its delayed write would contradict timestamp serial order: 1 < 2.
</details>

<details><summary>9. What is exchanged for more concurrency in multiversion methods?</summary>

Storage and management of multiple data versions and their metadata. Readers can use an appropriate older version, while writes and cleanup still require control.
</details>

<details><summary>10. What must a forced checkpoint not be confused with?</summary>

A commit of every active transaction. Uncommitted changes may have been flushed, so earlier log records can still be required to undo them.
</details>

<details><summary>11. Why does immediate update need undo, and why can it also need redo?</summary>

Uncommitted data can reach disk, requiring before-images for undo. Committed data may not have reached disk, requiring after-images for redo under a no-force policy.
</details>

<details><summary>12. Does a successful nested child have an independently permanent outcome?</summary>

In the lecture's nested model, no: its commit is tentative until the top-level transaction commits.
</details>

## Exam connections: reviewed 2023-2025 Q3

These are **worked study answers derived from the actual papers**, not an official marking scheme. The papers guide practice without replacing any lecture topics or predicting future exams. `ri(x)` means transaction `Ti` reads `x`; `wi(x)` means it writes `x`; `ci` and `ai` mean commit and abort.

| Paper and source | Parts and printed marks | Skill to practise |
|---|---|---|
| [2023 Q3](../Exampapers/COMP0178A7PE.pdf#page=4), PDF p4 / printed p3 | (a) 10, (b) 10, (c) 5 | Conflict graph, feasible deadlock trace, immediate/deferred recovery |
| [2024 Q3](../Exampapers/COMP0178A7PF.pdf#page=4), PDF p4 / printed p3 | (a) 10, (b) 10, (c) 5 | Conflict graph, timestamp trace, deadlock policies |
| [2025 Q3](../Exampapers/COMP0178A7PG.pdf#page=4), PDF pp4-5 / printed pp3-4 | (a)(i) 5, (a)(ii) 5, (b) 10, (c) 5 | 2PL versus deadlock, unrecoverable schedule, conflict graph |

**Method:** list every conflicting cross-transaction operation pair, explain each directed edge, then test the graph for a cycle. If acyclic, give a topological order; if cyclic, identify the cycle. A wait-for graph answers a different question: who is currently blocked by whose lock?

### 2023 Q3(a): an acyclic precedence graph

Source: [2023 paper, PDF p4](../Exampapers/COMP0178A7PE.pdf#page=4). The given schedule is:

```text
r1(x), r2(x), w2(y), r3(y), w1(x), w3(y)
```

| Earlier operation | Later operation | Edge and reason |
|---|---|---|
| `r2(x)` | `w1(x)` | `T2 -> T1`: the read must precede that write |
| `w2(y)` | `r3(y)` | `T2 -> T3`: the read follows T2's write |
| `w2(y)` | `w3(y)` | `T2 -> T3` again: preserve write order |

- The two initial reads of `x` do **not** conflict. There is no `T1 -> T2` edge from read/read operations.
- Graph: `T2 -> T1` and `T2 -> T3`, with no edge between `T1` and `T3`.
- **Answer:** conflict serializable, because the graph is acyclic. Both `T2, T1, T3` and `T2, T3, T1` are equivalent serial orders. The question does not determine a unique order.

### 2023 Q3(b)-(c): exhibit the wait, then explain recovery

Source: [2023 paper, PDF p4](../Exampapers/COMP0178A7PE.pdf#page=4).

**Q3(b): deadlock is possible.** A feasible interleaving, preserving each transaction's internal order, is:

1. T1 obtains an exclusive lock on A and reads A.
2. T2 obtains an exclusive lock on B.
3. T3 obtains an exclusive lock on C, reads C, and writes C.
4. T1 requests B and waits for T2.
5. T2 requests C and waits for T3.
6. T3 requests A and waits for T1.

The wait-for graph is `T1 -> T2 -> T3 -> T1`. Each holds a lock that the next transaction needs, and none reaches its unlock operation. A possible deadlock trace is enough; the claim is not that every interleaving deadlocks.

**Q3(c): storage and recovery, using the lecture's buffering assumptions.**

| Scheme | When database pages can receive an update | Consequence after a crash |
|---|---|---|
| Deferred update | Only after the transaction reaches commit | No UNDO of uncommitted database changes; REDO committed changes that have not reached storage |
| Immediate update | Before commit, when dirty buffers are flushed | UNDO uncommitted changes using before-images; REDO committed changes not yet durable |

- Immediate-update recovery needs write-ahead logging: the relevant log record must be durable before its changed data page is written. A durable commit record and the necessary redo information establish a committed transaction's recovery status.
- UNDO walks a losing transaction's updates backwards; REDO applies a winning transaction's changes forwards.
- **Qualification:** these UNDO/REDO requirements depend on the stated flushing policy. If a system forces every committed page to storage before acknowledging commit, REDO for those pages is unnecessary. The lecture's usual model allows committed pages still to be buffered.

### 2024 Q3(a): preserve conflicts on both x and y

Source: [2024 paper, PDF p4](../Exampapers/COMP0178A7PF.pdf#page=4). The schedule is:

```text
r1(x), r2(x), w1(x), r3(y), w2(y), r3(x)
```

| Conflicting pair | Edge |
|---|---|
| `r2(x)` before `w1(x)` | `T2 -> T1` |
| `w1(x)` before `r3(x)` | `T1 -> T3` |
| `r3(y)` before `w2(y)` | `T3 -> T2` |

- The graph contains `T2 -> T1 -> T3 -> T2`.
- **Answer:** not conflict serializable; no conflict-equivalent serial order exists.
- Do not ignore a read-before-write conflict because the read did not observe the later write. Preserving that fact is exactly why the edge is required.

### 2024 Q3(b)-(c): the first rejected timestamp operation

Source: [2024 paper, PDF p4](../Exampapers/COMP0178A7PF.pdf#page=4). Transaction numbers are timestamps: `TS(T1)=1`, `TS(T2)=2`, `TS(T3)=3`. Start with read and write timestamps of A and B at zero. `RTS` is the **maximum timestamp of any successful reader**, not merely the timestamp of the last reader to run.

| Step | Requested operation | Check and outcome | A after this step: `(RTS, WTS)` |
|---|---|---|---|
| 1 | `r1(A)` | `1 >= WTS(A)=0`: allow | `(1, 0)` |
| 2 | `r2(A)` | `2 >= WTS(A)=0`: allow | `(2, 0)` |
| 3 | `w1(A)` | `1 < RTS(A)=2`: reject and abort T1 | `(2, 0)` |

- **Answer:** the original sequence does not execute completely. The third operation attempts to install an older transaction's write after a newer transaction has already read A. Accepting it would violate the timestamp serial order `T1, T2, T3`.
- The abort occurs before the listed operations on B. Any restart of T1 with a new timestamp is a new scheduling decision; do not invent its placement and claim the original trace succeeded.
- The lecture's obsolete-write rule does not rescue this write. The read-timestamp check already fails, and `WTS(A)` is still zero.

**Q3(c): two supported options, with an optional third for comparison.**

1. **Prevention:** impose a policy that prevents circular waiting. For example, wait-die lets an older requester wait for a younger holder but aborts a younger requester facing an older holder. Restarting with its original timestamp helps avoid starvation.
2. **Detection and recovery:** periodically construct the wait-for graph; on finding a cycle, abort a chosen victim, roll back its effects, release its locks, and restart it. Choose victims fairly so the same transaction is not repeatedly starved.

A **timeout** is another option discussed in the lecture: abort a transaction after waiting too long. It is simple, but a slow transaction may time out even when no deadlock exists. Name two methods and explain their actions rather than only listing terms.

### 2025 Q3(a)-(b): serializability, deadlock, and recovery are separate checks

Source: [2025 paper, PDF p4](../Exampapers/COMP0178A7PG.pdf#page=4).

**Q3(a)(i), 1-2 sentence answer:** No non-serializable completed interleaving is possible under the stated exclusive locks and correct lock enforcement: both transactions acquire all their locks before releasing any, so they obey two-phase locking and guarantee conflict serializability.

- T2's `Unlock(x)` occurs before its later read/write of **y**, which is still locked; it acquires no new lock after that unlock. This satisfies basic 2PL.
- Commit points are not supplied. The lock sequences establish conflict serializability; they alone do not establish strictness or recoverability.
- This assumes the transactions themselves preserve the intended constraints when run serially, as standard serializability reasoning does.

**Q3(a)(ii), 1-2 sentence answer:** Yes: T1 can hold y while T2 holds x; T1 then waits for T2's x lock while T2 waits for T1's y lock. The wait-for cycle `T1 -> T2 -> T1` is a deadlock even though both follow 2PL.

**Q3(b), constructed example:** start with `x=100` and execute:

```text
T1: write x=80
T2: read x      # sees T1's uncommitted value 80
T2: commit
T1: abort
```

- This has the form `w1(x), r2(x), c2, a1`.
- T2 reads from T1, so recoverability requires T1 to commit before T2 commits. Instead T2 commits first, and T1 never commits.
- Undoing T1 removes the value on which T2 relied. T2 has already committed, creating the durability/rollback dilemma. The missing writer-before-reader commit order is what makes the schedule unrecoverable; a dirty read alone need not be unrecoverable if the reader waits for the writer to commit.
- It is possible for this schedule's read/write precedence graph to be acyclic (`T1 -> T2`). Serializability alone therefore does not guarantee recoverability.

### 2025 Q3(c): a two-transaction cycle inside a three-transaction graph

Source: [2025 paper, PDF p5](../Exampapers/COMP0178A7PG.pdf#page=5). The schedule is:

```text
r1(A), r2(B), r3(A), w3(B), w2(A)
```

| Conflicting pair | Edge |
|---|---|
| `r1(A)` before `w2(A)` | `T1 -> T2` |
| `r2(B)` before `w3(B)` | `T2 -> T3` |
| `r3(A)` before `w2(A)` | `T3 -> T2` |

- `T2 -> T3 -> T2` is a cycle. T1's extra edge does not remove it.
- **Answer:** not conflict serializable; there is no conflict-equivalent serial order.
- Read/read on A (`r1(A)` and `r3(A)`) creates no edge.

**Before an exam:** practise rebuilding all three papers' graphs without looking at the answers, identify the first failing timestamp check, draw a feasible wait-for cycle, and explain UNDO versus REDO under explicit storage assumptions. See also the broader [past-paper guide](../exam_guide.ipynb).